In [2]:
!pip install pyyaml pydantic langchain-core -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import yaml
import json
import re

from pathlib import Path
from pydantic import BaseModel, Field, ConfigDict
from typing import List, Optional

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

In [4]:
base_path = Path("metricmind_week2")

semantic_path = base_path / "semantic_layer"
output_path = base_path / "outputs"

semantic_path.mkdir(parents=True, exist_ok=True)
output_path.mkdir(parents=True, exist_ok=True)

print("Week 2 project folders created successfully.")
print("Base folder:", base_path)
print("Semantic layer:", semantic_path)
print("Outputs:", output_path)

Week 2 project folders created successfully.
Base folder: metricmind_week2
Semantic layer: metricmind_week2\semantic_layer
Outputs: metricmind_week2\outputs


In [5]:
week1_file = Path("metricmind_transformed_sales.csv")

if week1_file.exists():
    sales_model = pd.read_csv(week1_file)
    print("Week 1 sales data loaded successfully.")
else:
    print("Week 1 CSV not found.")
    print("Creating a sample sales_model for Week 2 demonstration.")

    sales_model = pd.DataFrame({
        "order_id": range(1001, 1011),
        "date": pd.date_range("2026-01-01", periods=10, freq="D"),
        "country": ["India"] * 10,
        "state": [
            "Andhra Pradesh",
            "Telangana",
            "Karnataka",
            "Tamil Nadu",
            "Kerala",
            "Andhra Pradesh",
            "Telangana",
            "Karnataka",
            "Tamil Nadu",
            "Kerala"
        ],
        "city": [
            "Amalapuram",
            "Hyderabad",
            "Bengaluru",
            "Chennai",
            "Kochi",
            "Vijayawada",
            "Warangal",
            "Mysuru",
            "Madurai",
            "Kollam"
        ],
        "revenue": [12000, 18000, 15000, 21000, 9000,
                    14000, 11000, 17000, 13000, 8000],
        "cost": [7000, 10000, 9000, 12000, 5000,
                 8000, 6000, 9500, 7000, 4500]
    })

print("Rows:", len(sales_model))
display(sales_model.head())

Week 1 sales data loaded successfully.
Rows: 15


,order_id,customer_id,product_id,order_date,quantity,discount,customer_name,segment,region,country,product_name,category,unit_price,gross_sales,discount_amount,net_sales
0,O001,C001,P001,2026-01-05,5,0.05,Acme Corporation,ENTERPRISE,NORTH,USA,Analytics Platform,SOFTWARE,1200.0,6000.0,300.0,5700.0
1,O002,C002,P003,2026-01-08,2,0.10,Globex Industries,ENTERPRISE,WEST,USA,Data Warehouse,CLOUD,2500.0,5000.0,500.0,4500.0
2,O003,C003,P004,2026-01-12,3,0.00,Initech,SMB,SOUTH,USA,BI Dashboard,ANALYTICS,1500.0,4500.0,0.0,4500.0
3,O004,C004,P005,2026-01-15,4,0.05,Stark Technologies,ENTERPRISE,EAST,USA,AI Assistant,AI,3000.0,12000.0,600.0,11400.0
4,O005,C005,P006,2026-01-20,2,0.10,Wayne Enterprises,ENTERPRISE,NORTH,USA,Data Pipeline,ENGINEERING,2200.0,4400.0,440.0,3960.0


In [6]:
print("Available columns:")
for column in sales_model.columns:
    print("-", column)

Available columns:
- order_id
- customer_id
- product_id
- order_date
- quantity
- discount
- customer_name
- segment
- region
- country
- product_name
- category
- unit_price
- gross_sales
- discount_amount
- net_sales


In [7]:
required_columns = [
    "date",
    "country",
    "state",
    "city",
    "revenue",
    "cost"
]

missing_columns = [
    column for column in required_columns
    if column not in sales_model.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)
    print("Week 2 will use the sample semantic-layer data structure.")
else:
    sales_model["date"] = pd.to_datetime(sales_model["date"])

print("Required semantic-layer columns checked.")

Missing columns: ['date', 'state', 'city', 'revenue', 'cost']
Week 2 will use the sample semantic-layer data structure.
Required semantic-layer columns checked.


In [8]:
cube_schema = {
    "cubes": [
        {
            "name": "sales",
            "sql_table": "public.sales",

            "dimensions": [
                {
                    "name": "date",
                    "sql": "date",
                    "type": "time"
                },
                {
                    "name": "country",
                    "sql": "country",
                    "type": "string"
                },
                {
                    "name": "state",
                    "sql": "state",
                    "type": "string"
                },
                {
                    "name": "city",
                    "sql": "city",
                    "type": "string"
                }
            ],

            "measures": [
                {
                    "name": "revenue",
                    "sql": "revenue",
                    "type": "sum"
                },
                {
                    "name": "cost",
                    "sql": "cost",
                    "type": "sum"
                }
            ]
        }
    ]
}

print("Cube.dev semantic layer created.")

Cube.dev semantic layer created.


In [9]:
cube_yaml_file = semantic_path / "sales.yml"

with open(cube_yaml_file, "w", encoding="utf-8") as file:
    yaml.dump(
        cube_schema,
        file,
        sort_keys=False,
        allow_unicode=True
    )

print("Cube YAML saved to:")
print(cube_yaml_file)

Cube YAML saved to:
metricmind_week2\semantic_layer\sales.yml


In [10]:
with open(cube_yaml_file, "r", encoding="utf-8") as file:
    cube_yaml = file.read()

print(cube_yaml)

cubes:
- name: sales
  sql_table: public.sales
  dimensions:
  - name: date
    sql: date
    type: time
  - name: country
    sql: country
    type: string
  - name: state
    sql: state
    type: string
  - name: city
    sql: city
    type: string
  measures:
  - name: revenue
    sql: revenue
    type: sum
  - name: cost
    sql: cost
    type: sum



In [11]:
sales_cube = cube_schema["cubes"][0]

dimensions = [
    dimension["name"]
    for dimension in sales_cube["dimensions"]
]

measures = [
    measure["name"]
    for measure in sales_cube["measures"]
]

print("Approved Dimensions:")
print(dimensions)

print("\nApproved Measures:")
print(measures)

Approved Dimensions:
['date', 'country', 'state', 'city']

Approved Measures:
['revenue', 'cost']


In [12]:
semantic_schema = {
    "cube": sales_cube["name"],
    "dimensions": dimensions,
    "measures": measures
}

semantic_schema

{'cube': 'sales',
 'dimensions': ['date', 'country', 'state', 'city'],
 'measures': ['revenue', 'cost']}

In [13]:
def validate_semantic_query(query):
    """
    Ensures that the generated query only uses
    dimensions and measures approved by the semantic layer.
    """

    errors = []

    for measure in query.get("measures", []):
        measure_name = measure.split(".")[-1]

        if measure_name not in measures:
            errors.append(
                f"Invalid measure: {measure_name}"
            )

    for dimension in query.get("dimensions", []):
        dimension_name = dimension.split(".")[-1]

        if dimension_name not in dimensions:
            errors.append(
                f"Invalid dimension: {dimension_name}"
            )

    for time_dimension in query.get("timeDimensions", []):
        dimension_name = time_dimension["dimension"].split(".")[-1]

        if dimension_name not in dimensions:
            errors.append(
                f"Invalid time dimension: {dimension_name}"
            )

    return errors


print("Semantic query validator created.")

Semantic query validator created.


In [14]:
class TimeDimension(BaseModel):
    model_config = ConfigDict(extra="forbid")

    dimension: str
    dateRange: Optional[List[str]] = None


class MetricMindQuery(BaseModel):
    model_config = ConfigDict(extra="forbid")

    measures: List[str] = Field(default_factory=list)
    dimensions: List[str] = Field(default_factory=list)
    timeDimensions: List[TimeDimension] = Field(default_factory=list)
    filters: List[dict] = Field(default_factory=list)


print("Strict JSON models created.")

Strict JSON models created.


In [15]:
test_query = MetricMindQuery(
    measures=["sales.revenue"],
    dimensions=["sales.state"],
    timeDimensions=[
        TimeDimension(
            dimension="sales.date",
            dateRange=["2026-01-01", "2026-12-31"]
        )
    ],
    filters=[]
)

print(test_query.model_dump_json(indent=2))

{
  "measures": [
    "sales.revenue"
  ],
  "dimensions": [
    "sales.state"
  ],
  "timeDimensions": [
    {
      "dimension": "sales.date",
      "dateRange": [
        "2026-01-01",
        "2026-12-31"
      ]
    }
  ],
  "filters": []
}


In [16]:
schema_text = f"""
Cube: {semantic_schema["cube"]}

Dimensions:
{chr(10).join("- " + d for d in dimensions)}

Measures:
{chr(10).join("- " + m for m in measures)}
"""

system_prompt = f"""
You are MetricMind, an enterprise analytics assistant.

Your job is to convert a user's business question into a
valid Cube.dev analytics query.

APPROVED SEMANTIC LAYER:

{schema_text}

RULES:

1. Use ONLY dimensions defined in the semantic layer.
2. Use ONLY measures defined in the semantic layer.
3. Never invent a metric.
4. Never invent a database column.
5. Never invent a dimension.
6. Revenue means sales.revenue.
7. Cost means sales.cost.
8. Date means sales.date.
9. Return only valid JSON.
10. Do not return Markdown.
11. Do not add explanations outside the JSON.

JSON structure:

{{
    "measures": [],
    "dimensions": [],
    "timeDimensions": [],
    "filters": []
}}
"""

print(system_prompt)


You are MetricMind, an enterprise analytics assistant.

Your job is to convert a user's business question into a
valid Cube.dev analytics query.

APPROVED SEMANTIC LAYER:


Cube: sales

Dimensions:
- date
- country
- state
- city

Measures:
- revenue
- cost


RULES:

1. Use ONLY dimensions defined in the semantic layer.
2. Use ONLY measures defined in the semantic layer.
3. Never invent a metric.
4. Never invent a database column.
5. Never invent a dimension.
6. Revenue means sales.revenue.
7. Cost means sales.cost.
8. Date means sales.date.
9. Return only valid JSON.
10. Do not return Markdown.
11. Do not add explanations outside the JSON.

JSON structure:

{
    "measures": [],
    "dimensions": [],
    "timeDimensions": [],
    "filters": []
}



In [17]:
metricmind_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}")
])

print("LangChain prompt created successfully.")

LangChain prompt created successfully.


In [18]:
def generate_query(question):
    """
    Demonstration orchestration layer.

    Converts common business questions into
    Cube-compatible JSON using only approved
    semantic-layer fields.
    """

    question_lower = question.lower()

    query = {
        "measures": [],
        "dimensions": [],
        "timeDimensions": [],
        "filters": []
    }

    # Measures
    if "revenue" in question_lower:
        query["measures"].append("sales.revenue")

    if "cost" in question_lower:
        query["measures"].append("sales.cost")

    # Dimensions
    if "country" in question_lower:
        query["dimensions"].append("sales.country")

    if "state" in question_lower:
        query["dimensions"].append("sales.state")

    if "city" in question_lower:
        query["dimensions"].append("sales.city")

    # Time
    if "2026" in question_lower:
        query["timeDimensions"].append({
            "dimension": "sales.date",
            "dateRange": [
                "2026-01-01",
                "2026-12-31"
            ]
        })

    # If no supported measure was found
    if not query["measures"]:
        query["measures"] = ["sales.revenue"]

    return query


print("Query generator created.")

Query generator created.


In [19]:
def agent_orchestration(inputs):
    question = inputs["question"]

    raw_query = generate_query(question)

    errors = validate_semantic_query(raw_query)

    if errors:
        return {
            "status": "error",
            "errors": errors
        }

    validated_query = MetricMindQuery(**raw_query)

    return {
        "status": "success",
        "query": validated_query.model_dump()
    }


metricmind_agent = RunnableLambda(agent_orchestration)

print("LangChain orchestration chain created.")

LangChain orchestration chain created.


In [20]:
question_1 = "Show revenue by state for 2026."

result_1 = metricmind_agent.invoke({
    "question": question_1
})

print(json.dumps(result_1, indent=2))

{
  "status": "success",
  "query": {
    "measures": [
      "sales.revenue"
    ],
    "dimensions": [
      "sales.state"
    ],
    "timeDimensions": [
      {
        "dimension": "sales.date",
        "dateRange": [
          "2026-01-01",
          "2026-12-31"
        ]
      }
    ],
    "filters": []
  }
}


In [21]:
questions = [
    "Show revenue by country.",
    "Show cost by state.",
    "Show revenue by city for 2026.",
    "Show revenue and cost by state.",
    "Show revenue for 2026."
]

for question in questions:

    result = metricmind_agent.invoke({
        "question": question
    })

    print("=" * 70)
    print("QUESTION:", question)
    print(json.dumps(result, indent=2))

QUESTION: Show revenue by country.
{
  "status": "success",
  "query": {
    "measures": [
      "sales.revenue"
    ],
    "dimensions": [
      "sales.country"
    ],
    "timeDimensions": [],
    "filters": []
  }
}
QUESTION: Show cost by state.
{
  "status": "success",
  "query": {
    "measures": [
      "sales.cost"
    ],
    "dimensions": [
      "sales.state"
    ],
    "timeDimensions": [],
    "filters": []
  }
}
QUESTION: Show revenue by city for 2026.
{
  "status": "success",
  "query": {
    "measures": [
      "sales.revenue"
    ],
    "dimensions": [
      "sales.city"
    ],
    "timeDimensions": [
      {
        "dimension": "sales.date",
        "dateRange": [
          "2026-01-01",
          "2026-12-31"
        ]
      }
    ],
    "filters": []
  }
}
QUESTION: Show revenue and cost by state.
{
  "status": "success",
  "query": {
    "measures": [
      "sales.revenue",
      "sales.cost"
    ],
    "dimensions": [
      "sales.state"
    ],
    "timeDimensions"

In [22]:
invalid_query = {
    "measures": ["sales.profit"],
    "dimensions": ["sales.state"],
    "timeDimensions": [],
    "filters": []
}

validation_result = validate_semantic_query(invalid_query)

if validation_result:
    print("INVALID QUERY BLOCKED")
    for error in validation_result:
        print("-", error)
else:
    print("Query is valid.")

INVALID QUERY BLOCKED
- Invalid measure: profit


In [23]:
def metricmind_response(question):
    """
    Main MetricMind entry point.
    """

    result = metricmind_agent.invoke({
        "question": question
    })

    response = {
        "question": question,
        "status": result["status"]
    }

    if result["status"] == "success":
        response["cube"] = "sales"
        response["query"] = result["query"]

    else:
        response["errors"] = result["errors"]

    return response


final_response = metricmind_response(
    "Show revenue by state for 2026."
)

print(
    json.dumps(
        final_response,
        indent=2
    )
)

{
  "question": "Show revenue by state for 2026.",
  "status": "success",
  "cube": "sales",
  "query": {
    "measures": [
      "sales.revenue"
    ],
    "dimensions": [
      "sales.state"
    ],
    "timeDimensions": [
      {
        "dimension": "sales.date",
        "dateRange": [
          "2026-01-01",
          "2026-12-31"
        ]
      }
    ],
    "filters": []
  }
}


In [24]:
week2_summary = {
    "project": "MetricMind",
    "week": 2,
    "component": "Semantic Layer + LangChain Agent",
    "cube": "sales",
    "dimensions": dimensions,
    "measures": measures,
    "strict_json": True,
    "schema_validation": True,
    "langchain_orchestration": True
}

summary_file = output_path / "week2_summary.json"

with open(summary_file, "w", encoding="utf-8") as file:
    json.dump(
        week2_summary,
        file,
        indent=2
    )

print("=" * 60)
print("METRICMIND WEEK 2 COMPLETED")
print("=" * 60)

print("\nCube:")
print("sales")

print("\nDimensions:")
for item in dimensions:
    print("-", item)

print("\nMeasures:")
for item in measures:
    print("-", item)

print("\nStrict JSON:", True)
print("Schema validation:", True)
print("LangChain orchestration:", True)

print("\nSemantic layer:")
print(cube_yaml_file)

print("\nWeek 2 summary:")
print(summary_file)

print("\nAll Week 2 components completed successfully.")

METRICMIND WEEK 2 COMPLETED

Cube:
sales

Dimensions:
- date
- country
- state
- city

Measures:
- revenue
- cost

Strict JSON: True
Schema validation: True
LangChain orchestration: True

Semantic layer:
metricmind_week2\semantic_layer\sales.yml

Week 2 summary:
metricmind_week2\outputs\week2_summary.json

All Week 2 components completed successfully.
